# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a reproducible workflow for loading, exploring, and processing the [FAIR^2 dataset](https://doi.org/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
This dataset is defined by a [Croissant schema](https://mlcommons.org/croissant/) and available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets, fields, and corresponding IDs for exploration
print("Available Record Sets:")
for rs in metadata.record_sets:
    print(f"- Name: {rs.name}\n  @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.id} (name: {field.name}, type: {field.data_type})")
    print("")

## 3. Data Extraction
Load data from each record set into a DataFrame for exploration. All references use `@id` fields for consistency.

In [ ]:
# Collect all record set @ids
record_sets = [rs.id for rs in metadata.record_sets]
print("Record Set IDs:", record_sets)

dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} rows for record set {record_set_id}")

# For demonstration, get the first available record set
primary_record_set_id = record_sets[0] if record_sets else None
if primary_record_set_id:
    print(f"Column names in DataFrame for {primary_record_set_id}:")
    print(dataframes[primary_record_set_id].columns.tolist())
    display(dataframes[primary_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply data cleaning and processing steps such as filtering records, normalizing numeric fields, and grouping by attributes. All field references use `@id` identifiers.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# --- Example using fields in the primary record set ---
rs = next((rs for rs in metadata.record_sets if rs.id == primary_record_set_id), None)
field_ids = [f.id for f in rs.fields] if rs is not None else []

# Select a representative numeric field for analysis (by @id)
numeric_field_id = None
for f in rs.fields:
    if f.data_type in ['Integer', 'Float', 'Number']:
        numeric_field_id = f.id
        break

if not numeric_field_id:
    print("No numeric field detected in record set.")
else:
    print(f"Numeric field selected: {numeric_field_id}")

    df = dataframes[primary_record_set_id]
    # Try to convert to numeric in case of CSV/object import
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Set a sample threshold; here we use mean if column exists
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the field
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, col_norm]].head())

    # Try grouping by a text/categorical field (first non-numeric)
    group_field_id = None
    for f in rs.fields:
        if f.data_type not in ['Integer', 'Float', 'Number']:
            group_field_id = f.id
            break
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = (
            filtered_df.groupby(group_field_id)[numeric_field_id, col_norm]
            .mean(numeric_only=True).reset_index()
        )
        print(f"Grouped data by {group_field_id} (showing means):")
        display(grouped_df.head())

## 5. Visualization

Let's visualize the distribution of a numeric field and the group-wise means (if possible).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouped, plot bars
    if 'grouped_df' in locals():
        plt.figure(figsize=(10,4))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Group mean of {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- Using `mlcroissant`, we loaded both metadata and records from the FAIR^2 mass spectrometry dataset using only `@id`-based referencing for record sets and fields.
- We reviewed available record sets and fields by their IDs and types, extracted data, conducted filtering and normalization of a primary numeric field, and visualized both distributions and group means.

This notebook provides a customizable template for further analysis of other Croissant datasets with robust, dataset-agnostic reference by `@id`.